# 05 — Grad-CAM Explainability

Visual explanations of model decisions using Gradient-weighted Class Activation Mapping:
- Per-class Grad-CAM grid
- Side-by-side: Original | Heatmap | Overlay
- Correct vs misclassified comparison
- Multi-class activation maps for a single image

In [ ]:
import os, sys, importlib

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
from src.data_manager import ensure_dataset

ensure_dataset()
config.ensure_dirs()

print(f"Images dir: {config.IMAGES_DIR} (exists: {os.path.isdir(config.IMAGES_DIR)})")
print(f"Outputs dir: {config.OUTPUTS_DIR}")

In [ ]:
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils import seed_everything, get_device
from src.dataset import DRDataset
from src.augmentations import get_valid_transforms
from src.model import build_model
from src.preprocessing import preprocess_image
from src.gradcam import GradCAM, get_target_layer, visualize_gradcam, visualize_gradcam_grid

seed_everything()
device = get_device()

## 1. Load Model and Data

In [ ]:
# Load model
model = build_model(pretrained=False)
ckpt_path = os.path.join(config.CHECKPOINTS_DIR, 'best_model_phase2.pth')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()
print(f'Model loaded (val QWK: {ckpt["metrics"]["kappa"]:.4f})')

In [ ]:
# Load test set
test_df = pd.read_csv(os.path.join(config.OUTPUTS_DIR, 'test_split.csv'))
valid_transforms = get_valid_transforms()
test_dataset = DRDataset(
    test_df, images_dir=config.IMAGES_DIR,
    transform=valid_transforms, preprocessing=True,
)
print(f'Test dataset: {len(test_dataset)} images')

## 2. Single Image Grad-CAM

In [ ]:
def find_image_path(images_dir, image_id):
    """Try common extensions and return the file path, or None."""
    for ext in ('.jpeg', '.jpg', '.png', '.JPEG', '.JPG'):
        p = os.path.join(images_dir, image_id + ext)
        if os.path.isfile(p):
            return p
    return None

# Pick one sample per class for detailed visualisation
for cls in range(config.NUM_CLASSES):
    cls_indices = test_df[test_df['level'] == cls].index.tolist()
    if not cls_indices:
        continue
    idx = cls_indices[0]
    
    # Get the original image for overlay
    img_id = test_df.iloc[idx]['image']
    img_path = find_image_path(config.IMAGES_DIR, img_id)
    if img_path is None:
        print(f'WARNING: Image not found for {img_id}, skipping class {cls}')
        continue
    original = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    preprocessed = preprocess_image(original)
    
    # Get normalised tensor
    img_tensor, label = test_dataset[idx]
    input_tensor = img_tensor.unsqueeze(0).to(device)
    
    print(f'\nClass {cls}: {config.CLASS_NAMES[cls]}')
    visualize_gradcam(
        model=model,
        image=preprocessed,
        input_tensor=input_tensor,
        device=device,
        target_class=cls,
        save_path=os.path.join(config.FIGURES_DIR, f'gradcam_class_{cls}.png'),
    )

## 3. Grad-CAM Grid (All Classes)

In [ ]:
visualize_gradcam_grid(
    model=model,
    dataset=test_dataset,
    device=device,
    n_per_class=3,
    save_path=os.path.join(config.FIGURES_DIR, 'gradcam_grid.png'),
)

## 4. Multi-Class Activation on a Single Image

Show what the model sees when asked about each of the 5 classes for the same image.

In [ ]:
# Use a Moderate DR image and show Grad-CAM for all 5 class targets
moderate_idx = test_df[test_df['level'] == 2].index.tolist()
if moderate_idx:
    idx = moderate_idx[0]
    img_id = test_df.iloc[idx]['image']
    img_path = find_image_path(config.IMAGES_DIR, img_id)
    original = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    preprocessed = preprocess_image(original)
    img_tensor, true_label = test_dataset[idx]
    input_tensor = img_tensor.unsqueeze(0).to(device)

    target_layer = get_target_layer(model)
    cam = GradCAM(model, target_layer)

    fig, axes = plt.subplots(1, config.NUM_CLASSES + 1, figsize=(4 * (config.NUM_CLASSES + 1), 4))
    fig.suptitle(f'Multi-Class Grad-CAM (True: {config.CLASS_NAMES[true_label]})', fontsize=16, y=1.02)

    axes[0].imshow(preprocessed)
    axes[0].set_title('Original', fontsize=11)
    axes[0].axis('off')

    for cls in range(config.NUM_CLASSES):
        heatmap = cam.generate_heatmap(input_tensor.clone(), target_class=cls)
        overlay = cam.overlay_heatmap(preprocessed, heatmap)
        axes[cls + 1].imshow(overlay)
        axes[cls + 1].set_title(config.CLASS_NAMES[cls], fontsize=11)
        axes[cls + 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(config.FIGURES_DIR, 'gradcam_multiclass.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 5. Correct vs Misclassified Comparison

In [ ]:
from src.evaluate import predict

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=config.PHASE1_BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY,
)
y_true, y_pred, y_proba = predict(model, test_loader, device)

correct_idx = np.where(y_true == y_pred)[0]
wrong_idx = np.where(y_true != y_pred)[0]

print(f'Correct: {len(correct_idx)} | Wrong: {len(wrong_idx)}')

In [ ]:
# Show Grad-CAM for 4 correct and 4 wrong predictions side by side
n_show = 4
target_layer = get_target_layer(model)

fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 10))
fig.suptitle('Grad-CAM: Correct (top) vs Misclassified (bottom)', fontsize=16, y=1.02)

for row, indices, title in [
    (0, correct_idx[:n_show], 'Correct'),
    (1, wrong_idx[:n_show], 'Misclassified'),
]:
    for col, idx in enumerate(indices):
        cam_obj = GradCAM(model, target_layer)
        img_tensor, label = test_dataset[idx]
        
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(config.IMG_STD) + np.array(config.IMG_MEAN)
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        input_t = img_tensor.unsqueeze(0).to(device)
        heatmap = cam_obj.generate_heatmap(input_t)
        overlay = cam_obj.overlay_heatmap(img_np, heatmap)
        
        axes[row, col].imshow(overlay)
        axes[row, col].set_title(
            f'True: {config.CLASS_NAMES[y_true[idx]]}\n'
            f'Pred: {config.CLASS_NAMES[y_pred[idx]]} ({y_proba[idx].max():.0%})',
            fontsize=10, color='green' if row == 0 else 'red',
        )
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'gradcam_correct_vs_wrong.png'), dpi=150, bbox_inches='tight')
plt.show()

## Summary

Grad-CAM reveals which retinal structures the model focuses on:
- **No DR**: model should show diffuse, low-activation maps
- **Mild/Moderate**: expect focus on microaneurysms and small haemorrhages
- **Severe**: larger haemorrhage regions, venous beading
- **Proliferative**: neovascularisation near the optic disc

These visualisations are critical for clinical trust and model debugging.